# Day 5 — Exploratory Data Analysis Part 2

This notebook extends the sales analysis into time, geography, customer behavior, and product performance. All metrics are calculated from the real UCI Online Retail dataset at runtime.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from src.data_cleaning import clean_online_retail
from src.eda import positive_sales_view

dataset = fetch_ucirepo(id=352)
df = clean_online_retail(dataset.data.features.copy())
sales_df = positive_sales_view(df)
sales_df['month'] = sales_df['invoice_date'].dt.to_period('M').astype(str)
sales_df['weekday'] = sales_df['invoice_date'].dt.day_name()


## Monthly trends
Monthly revenue, distinct orders, and distinct customers reveal changes in commercial activity over time.

In [ ]:
monthly = sales_df.groupby('month').agg(
    revenue=('revenue','sum'),
    orders=('invoice_no','nunique'),
    customers=('customer_id','nunique')
).reset_index()
display(monthly)

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
sns.lineplot(data=monthly, x='month', y='revenue', marker='o', ax=ax)
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Weekday purchasing behavior

In [ ]:
weekday_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday = sales_df.groupby('weekday').agg(revenue=('revenue','sum'), orders=('invoice_no','nunique')).reindex(weekday_order)
display(weekday)

In [ ]:
weekday['revenue'].plot(kind='bar', figsize=(10,5), title='Revenue by Day of Week')
plt.xlabel('Day')
plt.ylabel('Revenue')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Geographic performance
Country-level analysis is used instead of inventing a category field, because the source dataset contains `Country` but no formal product-category column.

In [ ]:
country = sales_df.groupby('country').agg(
    revenue=('revenue','sum'),
    orders=('invoice_no','nunique'),
    customers=('customer_id','nunique')
).sort_values('revenue', ascending=False)
display(country.head(15))

In [ ]:
country.head(10)['revenue'].sort_values().plot(kind='barh', figsize=(9,6), title='Top Countries by Revenue')
plt.xlabel('Revenue')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

## Customer purchasing behavior

In [ ]:
customer = sales_df.dropna(subset=['customer_id']).groupby('customer_id').agg(
    revenue=('revenue','sum'),
    orders=('invoice_no','nunique'),
    units=('quantity','sum')
).sort_values('revenue', ascending=False)
display(customer.describe().T)
display(customer.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=customer, x='orders', y='revenue', alpha=0.5, ax=ax)
ax.set_title('Customer Orders vs Revenue')
ax.set_xlabel('Distinct Orders')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()

## Interpretation guide

Use these tables and charts to identify sustained trends, geographic concentration, repeat purchasing, and customer-value patterns. Final business conclusions are intentionally deferred to Day 6 so they can be tied to observed evidence rather than assumptions.